# FFM_3_1 — mesh2z：SONICOM 的 latent 与端到端参照尺

**做什么**：用冻结的 FFM26 编码全部 399 个 SONICOM 受试者，导出 mesh2z 的回归目标；
再在有网格、且**不在 AE 测试集**的 164 人上，算出四把「dB 参照尺」——以后任何
mesh→z 的端到端结果都必须落在它们之间才有意义。

**不做什么**：不训练任何东西；**不对 AE 测试集算任何误差**。有网格的 36 个 AE-test 受试者
只编码、存 z，留作 mesh2z 的最终测试集，只在最后用一次。FFM26 的守护测试集也因此不被动用。

| 输入（Colab：Drive；本地：环境变量/默认路径） | |
|---|---|
| `HRTF_Checkpoints/FFM26_best.pth` | 四库模型，epoch 99 |
| `HRTF_Checkpoints/split_v21.json` | 必须与 checkpoint 里的划分一致 |
| `ELEC5020/SONICOM.zip` | SONICOM 的 npy_left / npy_right |
| `ELEC5020/SONICOM/mesh2z_samples/manifest.csv` | `sample_meshes.py` 的输出，只用它的受试者名单 |

**输出**：`HRTF_Checkpoints/mesh2z_sonicom_latents_v1.npz`

In [ ]:
# ==========================================================================
# Cell 0 — 环境 · 路径
# ==========================================================================
import os, sys, json, zipfile, hashlib, csv, re, warnings
from collections import Counter
warnings.filterwarnings('ignore')

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive'
    SAVE_DIR = f'{DRIVE}/HRTF_Checkpoints'
    os.system('pip install healpy -q')
    PATHS = dict(
        ckpt     = f'{SAVE_DIR}/FFM26_best.pth',
        split    = f'{SAVE_DIR}/split_v21.json',
        sonicom  = f'{DRIVE}/ELEC5020/SONICOM.zip',
        manifest = f'{DRIVE}/ELEC5020/SONICOM/mesh2z_samples/manifest.csv',
    )
    OUT = f'{SAVE_DIR}/mesh2z_sonicom_latents_v1.npz'
else:                                   # 本地 CPU 运行（推理不需要 GPU）
    H = os.path.expanduser('~/Documents')
    PATHS = dict(
        ckpt     = os.environ.get('FFM26_CKPT', f'{H}/26S2/ELEC5021/hrtf-GitHub/results/FFM26_best.pth'),
        split    = f'{H}/26S2/ELEC5021/hrtf-GitHub/analysis/split_v21.json',
        sonicom  = f'{H}/26S1/ELEC5020/SONICOM',          # 本地是已解压的目录
        manifest = f'{H}/26S1/ELEC5020/SONICOM/mesh2z_samples/manifest.csv',
    )
    OUT = os.environ.get('MESH2Z_OUT', f'{H}/26S1/ELEC5020/SONICOM/mesh2z_sonicom_latents_v1.npz')

for k, p in PATHS.items():
    print(f'{k:9s} {"OK " if os.path.exists(p) else "★缺失"} {p}')
assert all(os.path.exists(p) for p in PATHS.values()), '先把缺失的文件放好'

def find_dir(root, target):
    """递归找名为 target 的文件夹，跳过 __MACOSX，选 .npy 最多的那个"""
    c = []
    for dp, _, fns in os.walk(root):
        if '__MACOSX' in dp: continue
        if os.path.basename(dp) == target:
            c.append((sum(f.endswith('.npy') for f in fns), dp))
    c.sort(reverse=True)
    return c[0][1] if c else None

if PATHS['sonicom'].endswith('.zip'):
    SON_OUT = '/content/dataset/sonicom'            # ★ 每个库单独解压，不要共用目录
    os.makedirs(SON_OUT, exist_ok=True)
    if find_dir(SON_OUT, 'npy_left') is None:
        with zipfile.ZipFile(PATHS['sonicom']) as z: z.extractall(SON_OUT)
else:
    SON_OUT = PATHS['sonicom']
SON = dict(left=find_dir(SON_OUT, 'npy_left'), right=find_dir(SON_OUT, 'npy_right'),
           suffix=('_left_full.npy', '_right_full.npy'), did=2)
assert SON['left'] and SON['right'], '没找到 SONICOM 的 npy_left / npy_right'
print(f"\nSONICOM left  {SON['left']}  ({len(os.listdir(SON['left']))} 个文件)")
print(f"SONICOM right {SON['right']}  ({len(os.listdir(SON['right']))} 个文件)")

In [ ]:
# ==========================================================================
# Cell 1 — 配置
# ==========================================================================
CFG = dict(
    DID        = 2,          # FFM26: CARLab 0 / HUTUBS 1 / SONICOM 2 / SYMARE 3
    TAG        = 'SONICOM',
    NOTCH_HZ   = (6000, 12000),
    BATCH      = 16,
    EXPECT_MESH_PARTS = {'train': 147, 'val': 17, 'test': 36},   # sample_meshes.py, 2026-10-01
)
print(json.dumps(CFG, indent=1))

In [ ]:
# ==========================================================================
# Cell 2 - 模型定义（与 FFM_2_3 / FFM_2_6 逐字相同，构建时复制进来）
# ==========================================================================
import numpy as np
import torch
import torch.nn as nn
import healpy as hp


# ---------------------------------------------- spherical convolution
class OSLOConv(nn.Module):
    """Anisotropic convolution on HEALPix: each pixel plus its 8 neighbours, 9 independent weights."""
    def __init__(self, in_channels, out_channels, nside=16):
        super().__init__()
        self.npix = hp.nside2npix(nside)
        neighbors = hp.get_all_neighbours(nside, np.arange(self.npix))
        rf = np.concatenate([neighbors, np.arange(self.npix)[np.newaxis, :]], axis=0)
        rf[rf == -1] = self.npix
        self.register_buffer('neighborhood', torch.tensor(rf, dtype=torch.int64))
        self.weight = nn.Parameter(torch.empty(out_channels, in_channels, 9))
        self.bias = nn.Parameter(torch.empty(out_channels))
        nn.init.kaiming_uniform_(self.weight, a=np.sqrt(5))
        nn.init.zeros_(self.bias)

    def forward(self, x):
        B, C, _ = x.shape
        pad = torch.zeros(B, C, 1, device=x.device, dtype=x.dtype)
        g = torch.cat([x, pad], dim=2)[:, :, self.neighborhood]
        return torch.einsum('bcqn,ocq->bon', g, self.weight) + self.bias.view(1, -1, 1)


class OSLOResBlock(nn.Module):
    """Frequency-isolated residual block, operating on (B*F, C, N).
    * v2: ReLU -> GELU. HRTF log-magnitude is bipolar (negative means attenuation), so
      ReLU truncates the negative half; GELU is smooth, keeps negatives, and matches the fc layers."""
    def __init__(self, in_channels, out_channels, nside=16, num_groups=8):
        super().__init__()
        self.shortcut = (nn.Conv1d(in_channels, out_channels, 1)
                         if in_channels != out_channels else nn.Identity())
        self.conv1 = OSLOConv(in_channels, out_channels, nside)
        self.gn1 = nn.GroupNorm(num_groups, out_channels)
        self.conv2 = OSLOConv(out_channels, out_channels, nside)
        self.gn2 = nn.GroupNorm(num_groups, out_channels)
        self.act = nn.GELU()

    def forward(self, x):
        res = self.shortcut(x)
        out = self.act(self.gn1(self.conv1(x)))
        out = self.gn2(self.conv2(out))
        return self.act(out + res)


# ---------------------------------------------- per-dataset FiLM
class DatasetFiLM(nn.Module):
    """Learn one PER-FREQUENCY scale/shift curve for each dataset.

    ** Constraint 1: frequency only, NEVER direction.
      A per-frequency affine map is one EQ curve (93 dof per dataset). All it can absorb is
      "colouration common to the whole dataset and independent of direction" — loudspeaker and
      microphone response, equalisation convention, overall level. By construction it CANNOT
      express a direction-dependent difference, so population anatomy (ear-size distributions
      differ across databases) must still live in z. Contrast: subtracting a per-database mean
      over (frequency x direction) is 3072x93 dof, enough to remove the real population structure
      as well — exactly the wrong direction Craig pointed at. A factor of 3072 in expressive power.

    ** Constraint 2: centre across datasets.
      Without centring, log_gamma/beta learn a global EQ that cancels against the network's own
      output gain — a redundant degree of freedom. After the cross-database mean is removed FiLM
      expresses only RELATIVE differences; the absolute part stays in the network. Exactly the identity when num_datasets=1.
    """
    def __init__(self, num_datasets, num_bins):
        super().__init__()
        self.num_datasets = num_datasets
        self.log_gamma = nn.Parameter(torch.zeros(num_datasets, num_bins))
        self.beta = nn.Parameter(torch.zeros(num_datasets, num_bins))

    def params(self, dataset_id):
        lg = self.log_gamma - self.log_gamma.mean(0, keepdim=True)
        bt = self.beta - self.beta.mean(0, keepdim=True)
        return torch.exp(lg[dataset_id]).unsqueeze(-1), bt[dataset_id].unsqueeze(-1)

    def forward(self, x, dataset_id):
        g, b = self.params(dataset_id)
        return x * g + b

    def inverse(self, x, dataset_id):
        g, b = self.params(dataset_id)
        return (x - b) / g


# ---------------------------------------------- encoder / decoder

class Isolated_OSLO_Encoder(nn.Module):
    """Encode each frequency bin independently -> latent (B, F, latent_dim)"""
    def __init__(self, nside=16, latent_dim_per_freq=64):
        super().__init__()
        self.npix = hp.nside2npix(nside)
        self.latent_dim = latent_dim_per_freq
        self.enc1 = OSLOResBlock(1, 16, nside)
        self.enc2 = OSLOResBlock(16, 32, nside)
        self.fc_encode = nn.Sequential(
            nn.Flatten(), nn.Linear(32 * self.npix, 512), nn.GELU(),
            nn.Linear(512, self.latent_dim))

    def forward(self, x):
        B, F, N = x.shape
        z = self.fc_encode(self.enc2(self.enc1(x.reshape(B * F, 1, N))))
        return z.reshape(B, F, self.latent_dim)


class Conditioned_OSLO_Decoder(nn.Module):
    """FFM frequency-conditioned decoder + per-dataset FiLM on the output"""
    def __init__(self, nside=16, latent_dim_per_freq=64, freq_embed_dim=32,
                 num_datasets=1, num_bins=93):
        super().__init__()
        self.npix = hp.nside2npix(nside)
        self.latent_dim = latent_dim_per_freq
        self.freq_embed_dim = freq_embed_dim
        self.ffm_B = nn.Parameter(torch.randn(freq_embed_dim // 2))
        self.fc_decode = nn.Sequential(
            nn.Linear(self.latent_dim + freq_embed_dim, 512), nn.GELU(),
            nn.Linear(512, 32 * self.npix))
        self.dec1 = OSLOResBlock(32, 16, nside)
        self.dec_final = OSLOConv(16, 1, nside)
        self.film = DatasetFiLM(num_datasets, num_bins)

    def get_freq_embedding(self, num_freqs, device):
        f = torch.arange(num_freqs, device=device).float() / (num_freqs - 1)
        proj = 2 * np.pi * f.unsqueeze(1) * self.ffm_B.unsqueeze(0)
        return torch.cat([torch.sin(proj), torch.cos(proj)], dim=1)

    def forward(self, z, dataset_id=None):
        B, F, _ = z.shape
        fe = self.get_freq_embedding(F, z.device).unsqueeze(0).expand(B, -1, -1)
        h = torch.cat([z, fe], dim=-1).reshape(B * F, -1)
        h = self.fc_decode(h).reshape(B * F, 32, self.npix)
        out = self.dec_final(self.dec1(h)).reshape(B, F, self.npix)
        if dataset_id is not None:
            out = self.film(out, dataset_id)
        return out


class FFM_OSLO_Autoencoder(nn.Module):
    """One SHARED-parameter inverse FiLM at the encoder input — zero extra parameters.

    With FiLM on the decoder output only, the colouration stays in x, the encoder takes it in, and it
    lands in z — but z is exactly what the downstream geometry->latent stage needs. Output-side FiLM
    only makes the reconstruction loss line up; it does not clean z.

    Here     x_norm = (x - beta)/gamma        <- remove colouration
             z      = encoder(x_norm)
             out    = gamma * decoder(z) + beta   <- back to the original space, loss still on raw

    The SAME FiLM parameters are used. Together with centring this also kills the trivial degenerate
    direction "scale gamma up, scale the network down". Set tie_encoder_film=False to disable.
    """
    def __init__(self, nside=16, latent_dim_per_freq=64, freq_embed_dim=32,
                 num_datasets=1, num_bins=93, tie_encoder_film=True):
        super().__init__()
        self.encoder = Isolated_OSLO_Encoder(nside, latent_dim_per_freq)
        self.decoder = Conditioned_OSLO_Decoder(nside, latent_dim_per_freq,
                                                freq_embed_dim, num_datasets, num_bins)
        self.tie_encoder_film = tie_encoder_film

    def forward(self, x, dataset_id=None):
        if dataset_id is not None and self.tie_encoder_film:
            x = self.decoder.film.inverse(x, dataset_id)
        z = self.encoder(x)
        return self.decoder(z, dataset_id), z


print('model defined (FFM + FiLM)')

In [ ]:
# ==========================================================================
# Cell 3 — 受试者：manifest × split_v21.json
#
# ★ 编号格式两边不同：manifest 是 P0001，split 是 SON_P0001。按完整编号匹配，
#   且断言每一个都找到了 —— 「两边都没找到」不能被当成「没有泄漏」。
# ==========================================================================
SPLIT = json.load(open(PATHS['split']))
part_of = {s: p for p in ('train', 'val', 'test') for s in SPLIT['SONICOM'][p]}
assert len(part_of) == 399 and all(re.fullmatch(r'SON_P\d{4}', s) for s in part_of)

with open(PATHS['manifest'], newline='') as f:
    MESH_IDS = {f"SON_{r['subject']}" for r in csv.DictReader(f)}
missing = sorted(MESH_IDS - set(part_of))
assert not missing, f'{len(missing)} 个有网格的人不在 AE 里: {missing[:5]}'

sfxL, sfxR = SON['suffix']
on_disk = {f[:-len(sfxL)] for f in os.listdir(SON['left']) if f.endswith(sfxL)} & \
          {f[:-len(sfxR)] for f in os.listdir(SON['right']) if f.endswith(sfxR)}
assert set(part_of) <= on_disk, f'{len(set(part_of) - on_disk)} 个 AE 受试者没有双耳 npy'

SUBJ = [dict(sid=s, ae_part=part_of[s], has_mesh=s in MESH_IDS) for s in sorted(part_of)]
got = dict(Counter(s['ae_part'] for s in SUBJ if s['has_mesh']))
print(f"SONICOM {len(SUBJ)} 人 | 有网格 {len(MESH_IDS)} 人，按 AE 分区 {got}")
assert got == CFG['EXPECT_MESH_PARTS'], f"与 sample_meshes.py 时的交集不一致: {got}"

# 参照尺只在【有网格 且 不在 AE-test】的人上算
EVAL = [s for s in SUBJ if s['has_mesh'] and s['ae_part'] != 'test']
LOCKED = [s for s in SUBJ if s['has_mesh'] and s['ae_part'] == 'test']
SUBJ_HASH = hashlib.md5(','.join(s['sid'] for s in SUBJ).encode()).hexdigest()[:12]
MESH_HASH = hashlib.md5(','.join(sorted(MESH_IDS)).encode()).hexdigest()[:12]
print(f'参照尺 {len(EVAL)} 人 | 锁定（mesh2z 最终测试）{len(LOCKED)} 人 | hash {SUBJ_HASH}/{MESH_HASH}')

In [ ]:
# ==========================================================================
# Cell 4 — 载入 FFM26，编码全部 399 人
#
# ★ encoder.forward 只收 x。数据集条件化是【编码前的逆 FiLM】，做在
#   FFM_OSLO_Autoencoder.forward 里。直接 model.encoder(x) 会把测量染色编进 z。
# ==========================================================================
import numpy as np, torch, healpy as hp

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ckpt = torch.load(PATHS['ckpt'], map_location=device, weights_only=False)
n_ds, n_bins = ckpt['model_state_dict']['decoder.film.beta'].shape
print(f"checkpoint: epoch {ckpt['epoch']+1} | val {ckpt['val_rmse']:.4f} dB | {n_ds} 库 / {n_bins} bin | {device}")
assert n_bins == 93 and n_ds == 4, '需要四库 93-bin 的 FFM26'
assert ckpt['split']['SONICOM'] == SPLIT['SONICOM'], 'checkpoint 里的 SONICOM 划分和磁盘上的不一致'
CKPT_INFO = dict(epoch=int(ckpt['epoch']) + 1, val_rmse=float(ckpt['val_rmse']),
                 file=os.path.basename(PATHS['ckpt']))

model = FFM_OSLO_Autoencoder(nside=16, latent_dim_per_freq=64, freq_embed_dim=32,
                             num_datasets=n_ds, num_bins=n_bins, tie_encoder_film=True).to(device)
model.load_state_dict(ckpt['model_state_dict']); model.eval()
del ckpt
FREQS = np.arange(n_bins) * 44100 / 256
NOTCH = (FREQS >= CFG['NOTCH_HZ'][0]) & (FREQS <= CFG['NOTCH_HZ'][1])

_th, _ph = hp.pix2ang(16, np.arange(hp.nside2npix(16)))
MIRROR = hp.ang2pix(16, _th, (-_ph) % (2 * np.pi))
assert np.array_equal(MIRROR[MIRROR], np.arange(len(MIRROR))), '镜像索引不是对合'

def load_ear(sid, side):
    sfx = SON['suffix'][0 if side == 'left' else 1]
    x = np.load(os.path.join(SON[side], f'{sid}{sfx}')).astype(np.float32)
    assert x.shape == (129, 3072), f'{sid} {side} 形状 {x.shape}'
    x = x[:n_bins]
    return x[:, MIRROR] if side == 'right' else x

DID_T = lambda n: torch.full((n,), CFG['DID'], device=device, dtype=torch.long)

def encode(x_np):
    xt = torch.from_numpy(x_np).to(device)
    xn = model.decoder.film.inverse(xt, DID_T(len(xt))) if model.tie_encoder_film else xt
    return model.encoder(xn)

def decode(z_t):
    return model.decoder(z_t, DID_T(len(z_t)))

Z = np.zeros((len(SUBJ), 2, n_bins, 64), np.float32)      # [subject, side(L,R), bin, dim]
with torch.no_grad():
    x0 = load_ear(SUBJ[0]['sid'], 'left')[None]
    assert torch.equal(encode(x0), model(torch.from_numpy(x0).to(device), DID_T(1))[1]), \
        'encode() 与 model.forward 的 z 不一致'
    print('encode() 与 model.forward 逐位一致 ✓')
    for b in range(0, len(SUBJ), CFG['BATCH']):
        chunk = SUBJ[b:b + CFG['BATCH']]
        X = np.stack([load_ear(s['sid'], side) for s in chunk for side in ('left', 'right')])
        Z[b:b + len(chunk)] = encode(X).float().cpu().numpy().reshape(len(chunk), 2, n_bins, 64)
print(f'Z {Z.shape}，有限值 {np.isfinite(Z).all()}')

In [ ]:
# ==========================================================================
# Cell 5 — latent 诊断（先测，不要照抄历史数字）
# ==========================================================================
mesh_m = np.array([s['has_mesh'] for s in SUBJ])
Zf = Z.reshape(len(SUBJ), 2, -1)

# ① side 方向占多少方差（v1 71.7%，v2.3 实测 1.4%）。远超 5% 说明镜像没生效
v = Zf[:, 1].mean(0) - Zf[:, 0].mean(0); v /= np.linalg.norm(v)
Zall = Zf.reshape(-1, Zf.shape[-1]); Zc = Zall - Zall.mean(0)
side_frac = ((Zc @ v) ** 2).sum() / (Zc ** 2).sum()
print(f'① side 方向占 latent 方差 {side_frac:.2%}')

# ② 同一人左右耳的相似度 —— 决定「有效 N 按人算」
Zs = Zf - Zf.mean((0, 1))
cos_same = np.sum(Zs[:, 0] * Zs[:, 1], 1) / np.linalg.norm(Zs[:, 0], axis=1) / np.linalg.norm(Zs[:, 1], axis=1)
rng = np.random.default_rng(0); perm = rng.permutation(len(SUBJ))
cos_diff = np.sum(Zs[:, 0] * Zs[perm, 1], 1) / np.linalg.norm(Zs[:, 0], axis=1) / np.linalg.norm(Zs[perm, 1], axis=1)
print(f'② 左右耳 latent 余弦相似：同一人 {np.median(cos_same):.3f} | 随机配对 {np.median(cos_diff):.3f}'
      '  -> 统计单位是【人】，不是耳')

# ③ 有网格的 200 人是不是一个有偏子集（扫描质量差的 100 人被剔除了）
nrm = np.linalg.norm(Zs.mean(1), axis=1)
from scipy import stats
t = stats.mannwhitneyu(nrm[mesh_m], nrm[~mesh_m])
print(f'③ ||z_sub||：有网格 {np.median(nrm[mesh_m]):.1f} vs 无网格 {np.median(nrm[~mesh_m]):.1f}，'
      f'Mann-Whitney p={t.pvalue:.3f}  ({"无可测差异" if t.pvalue > .05 else "★有差异，写结果时要说明选择偏差"})')

In [ ]:
# ==========================================================================
# Cell 6 — 四把 dB 参照尺（只在 EVAL 的 164 人上，AE-test 一律不碰）
#
#   ① AE 重建         decode(z_ear)              —— HRTF 进、HRTF 出；mesh2z 不可能比它好
#   ② 双耳平均上限    decode(z_sub) 对比每只耳   —— mesh2z 预测的是【人】的 z，这是它的真实上限
#   ③ 均值 latent     decode(训练人群 z_sub 均值) —— 一个 R²=0 的预测器在 dB 上的样子
#   ④ 均值 HRTF       训练耳的平均 HRTF          —— 文献该报而常不报的群体均值基线
#
# 口径：每人 = 两只耳 × 93 bin × 3072 方向上的合并 RMSE；总体 = 各人平方的均值再开方
#       （与 evaluate() 相同的 pooled 口径）。另报 6–12 kHz。
# 「训练人群」= AE-train 的全部 299 个 SONICOM 受试者（无论有没有网格）。
# ==========================================================================
tr_m = np.array([s['ae_part'] == 'train' for s in SUBJ])

MU_X = np.zeros((n_bins, 3072), np.float64)
for s in (s for s in SUBJ if s['ae_part'] == 'train'):
    for side in ('left', 'right'):
        MU_X += load_ear(s['sid'], side)
MU_X /= 2 * tr_m.sum()

Z_sub = Z.mean(1)                                            # (N, 93, 64)
with torch.no_grad():
    MU_Z_DEC = decode(torch.from_numpy(Z_sub[tr_m].mean(0, keepdims=True)).to(device))[0].cpu().numpy()

def sq(err, mask=None):
    """err: (93 bin, 3072 方向)；mask 选频率，作用在第 0 维"""
    assert err.shape == (n_bins, 3072), err.shape
    e = err if mask is None else err[mask]
    return float((e.astype(np.float64) ** 2).mean())

idx_of = {s['sid']: i for i, s in enumerate(SUBJ)}
ROWS = []
with torch.no_grad():
    for s in EVAL:
        i = idx_of[s['sid']]
        X = np.stack([load_ear(s['sid'], 'left'), load_ear(s['sid'], 'right')])
        rec = decode(torch.from_numpy(Z[i]).to(device)).cpu().numpy()
        avg = decode(torch.from_numpy(Z_sub[i:i+1]).to(device)).cpu().numpy()
        row = dict(sid=s['sid'], ae_part=s['ae_part'])
        for name, P in (('recon', rec), ('ear_avg', np.repeat(avg, 2, 0)),
                        ('mean_z', np.repeat(MU_Z_DEC[None], 2, 0)), ('mean_hrtf', np.repeat(MU_X[None], 2, 0))):
            row[name] = np.mean([sq(P[k] - X[k]) for k in range(2)])
            row[name + '_notch'] = np.mean([sq(P[k] - X[k], NOTCH) for k in range(2)])
        ROWS.append(row)

REF = {}
print(f"{'参照尺（N=%d 人，pooled RMSE, dB）' % len(ROWS):34s}{'全频 0–15.85k':>14s}{'6–12 kHz':>12s}")
for name, label in (('recon', '① AE 重建'), ('ear_avg', '② 双耳平均上限'),
                    ('mean_z', '③ 均值 latent 解码'), ('mean_hrtf', '④ 均值 HRTF 基线')):
    a = np.sqrt(np.mean([r[name] for r in ROWS])); b = np.sqrt(np.mean([r[name + '_notch'] for r in ROWS]))
    REF[name] = dict(all=a, notch=b)
    print(f'{label:30s}{a:14.3f}{b:12.3f}')
print('\n以后任何 mesh→z 的端到端结果都应落在 ② 与 ④ 之间：')
print('  靠近 ④ = 几何没有提供超过「人群平均」的信息；靠近 ② = 几何几乎完全决定了这个人的 z。')
print(f"  可用于改进的空间：{REF['mean_hrtf']['all'] - REF['ear_avg']['all']:.2f} dB（全频），"
      f"{REF['mean_hrtf']['notch'] - REF['ear_avg']['notch']:.2f} dB（6–12 kHz）")

In [ ]:
# ==========================================================================
# Cell 7 — 导出缓存
# ==========================================================================
np.savez_compressed(
    OUT,
    sid=np.array([s['sid'] for s in SUBJ]),
    ae_part=np.array([s['ae_part'] for s in SUBJ]),
    has_mesh=mesh_m,
    Z_ear=Z.astype(np.float32),                  # (399, 2[L,R-mirrored], 93, 64)
    Z_sub=Z_sub.astype(np.float32),              # (399, 93, 64)  —— mesh2z 的回归目标
    mu_hrtf_train=MU_X.astype(np.float32),       # (93, 3072)     AE-train 299 人的均值 HRTF
    freqs=FREQS, did=CFG['DID'],
    ref_rows=json.dumps(ROWS), ref=json.dumps(REF),
    eval_sids=np.array([s['sid'] for s in EVAL]), locked_sids=np.array([s['sid'] for s in LOCKED]),
    side_frac=side_frac, subj_hash=SUBJ_HASH, mesh_hash=MESH_HASH, ckpt=json.dumps(CKPT_INFO),
)
print(f'-> {OUT}  ({os.path.getsize(OUT)/1e6:.1f} MB)')
print(f'   Z_sub {Z_sub.shape} | 参照尺 {len(EVAL)} 人 | 锁定 {len(LOCKED)} 人 | hash {SUBJ_HASH}/{MESH_HASH}')

## 交付前检查

1. **Cell 4**：`n_ds == 4`、划分一致、`encode()` 与 `model.forward` 逐位一致，三条断言都要过。
2. **Cell 5 ①**：side 方差应在几个百分点以内。几十个百分点说明镜像坏了。
3. **Cell 5 ③**：有网格的 200 人如果和其余 199 人有可测差异，写结果时必须说明——
   被剔除的是耳廓扫描失真的人。
4. **Cell 6**：③「均值 latent 解码」应当接近 ④「均值 HRTF」。差很多说明解码器在
   latent 均值处不线性，后面端到端结果要以 ④ 为基线、不能以 ③ 为基线。
5. **锁定的 36 人**在这里没有任何误差被计算。它们的第一次、也是唯一一次评估在
   mesh2z 的最终测试里。